In [1]:
import numpy as np
import torch
import h5py
import os
from scipy.signal import medfilt2d

def preprocess_real_world_data(file_path, save_name="real_eval_dataset.h5"):
    # --- 1. 参数对齐 (严格匹配 v6.5 仿真参数) ---
    fs = 500e3
    step_s = 0.05
    samples_per_step = int(fs * step_s)  # 25000点
    num_channels = 10
    window_size = 12                     # 历史窗口
    sync_len = 2000                      # 需切除的同步头长度

    print(f"🛠️ 启动实测数据流水线预处理...")
    
    # --- 2. 加载与初步清洗 ---
    raw_iq = np.fromfile(file_path, dtype=np.complex64)
    # 核心：切除开头的巴克码同步头
    iq_clean = raw_iq[sync_len:] 
    
    num_steps = len(iq_clean) // samples_per_step
    # 截取整数倍步长
    iq_steps = iq_clean[:num_steps * samples_per_step].reshape(num_steps, samples_per_step)
    
    # --- 3. 态势特征提取 (FFT 能量映射) ---
    # 将时域 IQ 转换为 10 信道频率功率
    situation_matrix = []
    for step in iq_steps:
        # 使用 FFT 获取功率谱
        fft_res = np.abs(np.fft.fft(step, n=1024))
        # 简化版：将 FFT 结果均匀划分为 10 个频率块并取能量均值
        freq_bins = np.array_split(np.fft.fftshift(fft_res), num_channels)
        energies = [np.mean(b) for b in freq_bins]
        situation_matrix.append(energies)
    
    situation_matrix = np.array(situation_matrix)
    
    # 归一化：将能量缩放至 [0, 1] 范围内
    sit_min, sit_max = situation_matrix.min(), situation_matrix.max()
    situation_matrix = (situation_matrix - sit_min) / (sit_max - sit_min + 1e-9)
    
    # 执行中值滤波：消除实测中残留的微小 U/S 干扰毛刺
    situation_matrix = medfilt2d(situation_matrix, kernel_size=(3, 1))

    # --- 4. 制作滑动窗口数据集 (Batch, 12, 10) ---
    X_test = []
    for i in range(len(situation_matrix) - window_size):
        X_test.append(situation_matrix[i : i + window_size])
    
    X_test = np.array(X_test)
    print(f"✅ 预处理完成！最终推理张量形状: {X_test.shape}")

    # --- 5. 保存供模型推理 ---
    with h5py.File(save_name, 'w') as f:
        f.create_dataset("X_eval", data=X_test)
        f.create_dataset("full_sit_matrix", data=situation_matrix)
    
    return X_test, situation_matrix

# 执行预处理
if __name__ == "__main__":
    # 请确保文件名与你录制的一致
    RAW_FILE = "/root/autodl-tmp/validate/0218/Prediction/Receive_datasets/rx_capture_v6_5_robust.dat"
    X_eval, full_sit = preprocess_real_world_data(RAW_FILE)

🛠️ 启动实测数据流水线预处理...
✅ 预处理完成！最终推理张量形状: (3987, 12, 10)
